# MML Workshop – Image Colorization

Finálna verzia notebooku pre lokálny projekt.

Notebook je nastavený tak, aby **zbytočne netrénoval znova**:

- ak už existuje finálny model `best.pt`, tréning sa preskočí,
- ak už existuje AB codebook, jeho generovanie sa preskočí,
- ak už existuje filtrovaný dataset, filtrovanie sa preskočí,
- demo sa iba vygeneruje/zobrazí z existujúceho modelu.

Finálny smer projektu:

```text
grayscale / L channel
→ LAB colour space
→ learned AB codebook
→ CNN encoder-decoder model
→ colour-bin classification + AB residual refinement
```


## 1. Nastavenie projektu

Notebook sa pokúsi nájsť koreň projektu automaticky. Ak by zlyhal, nastav `PROJECT_DIR` ručne na:

```text
D:\unicorn\mml_workshop
```


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import random
import shutil

def find_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path(r"D:\unicorn\mml_workshop"),
    ]

    for c in candidates:
        if (c / "scripts" / "train_codebook_colorizer.py").exists():
            return c.resolve()

    raise RuntimeError(
        "Project directory not found. Open this notebook from the project or set PROJECT_DIR manually."
    )

PROJECT_DIR = find_project_dir()
os.chdir(PROJECT_DIR)

print("Project directory:", PROJECT_DIR)
print("Current working directory:", Path.cwd())


## 2. Kontrola CUDA / PyTorch

Táto bunka len overí, či lokálny PyTorch vidí GPU. Netrenuje model.


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. Training would be very slow.")


## 3. Cesty a prepínače

Dôležité prepínače:

- `TRAIN_IF_MISSING = True` znamená: trénuj iba vtedy, ak finálny checkpoint neexistuje.
- `FORCE_RETRAIN = False` znamená: nikdy netrénuj znova, ak už `best.pt` existuje.
- `FORCE_REBUILD_CODEBOOK = False` znamená: negeneruj codebook znova, ak už existuje.


In [ ]:
from pathlib import Path

ORIGINAL_DATASET = Path("data/dataset")
FILTERED_DATASET = Path("data/dataset_filtered")

CODEBOOK = Path("models/codebook_filtered/ab_codebook_256_filtered.npz")
MODEL_DIR = Path("models/codebook_colorizer_filtered_v3")
BEST_MODEL = MODEL_DIR / "best.pt"

SAMPLE_DIR = Path("outputs/codebook_samples_filtered_v3")
FINAL_DEMO_DIR = Path("outputs/final_v3_demo")
FINAL_DEMO_GRID = FINAL_DEMO_DIR / "final_v3_random_grid.png"

TRAIN_IF_MISSING = True
FORCE_RETRAIN = False
FORCE_REBUILD_CODEBOOK = False
FORCE_REGENERATE_DEMO = True

print("Original dataset:", ORIGINAL_DATASET, ORIGINAL_DATASET.exists())
print("Filtered dataset:", FILTERED_DATASET, FILTERED_DATASET.exists())
print("Codebook:", CODEBOOK, CODEBOOK.exists())
print("Best model:", BEST_MODEL, BEST_MODEL.exists())
print("Final demo:", FINAL_DEMO_GRID, FINAL_DEMO_GRID.exists())


## 4. Dataset status

Tu iba vypíšeme počty obrázkov. Ak filtrovaný dataset neexistuje, ďalšia bunka ho vie vytvoriť.


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(path):
    path = Path(path)
    if not path.exists():
        return 0
    return len([p for p in path.rglob("*") if p.suffix.lower() in IMAGE_EXTS])

print("Original images:", count_images(ORIGINAL_DATASET))
print("Filtered images:", count_images(FILTERED_DATASET))


## 5. Vytvorenie filtrovaného datasetu, iba ak chýba

Používa sa LAB chroma filter. Ak `data/dataset_filtered` už existuje a obsahuje obrázky, bunka nič nekopíruje.


In [ ]:
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
import shutil

THRESHOLD = 8.0

if FILTERED_DATASET.exists() and count_images(FILTERED_DATASET) > 0:
    print("Filtered dataset already exists. Skipping filtering.")
    print("Filtered images:", count_images(FILTERED_DATASET))
else:
    if not ORIGINAL_DATASET.exists():
        raise RuntimeError(f"Original dataset not found: {ORIGINAL_DATASET}")

    FILTERED_DATASET.mkdir(parents=True, exist_ok=True)
    paths = [p for p in ORIGINAL_DATASET.rglob("*") if p.suffix.lower() in IMAGE_EXTS]

    kept = 0
    removed = 0
    rows = []

    for p in paths:
        try:
            img = Image.open(p).convert("RGB").resize((256, 256), Image.BICUBIC)
            rgb = np.asarray(img).astype(np.float32) / 255.0
            lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
            ab = lab[:, :, 1:3]
            chroma = np.sqrt((ab ** 2).sum(axis=2))
            mean_c = float(chroma.mean())

            rows.append(f"{mean_c:.2f}\t{p}")

            if mean_c >= THRESHOLD:
                rel = p.relative_to(ORIGINAL_DATASET)
                out = FILTERED_DATASET / rel
                out.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, out)
                kept += 1
            else:
                removed += 1

        except Exception as e:
            removed += 1
            rows.append(f"ERROR\t{p}\t{e}")

    Path("outputs").mkdir(exist_ok=True)
    Path("outputs/dataset_chroma_report.txt").write_text("\n".join(rows), encoding="utf-8")

    print("Source images:", len(paths))
    print("Kept:", kept)
    print("Removed low-chroma:", removed)
    print("Threshold mean chroma:", THRESHOLD)
    print("Filtered dataset:", FILTERED_DATASET)


## 6. AB codebook, iba ak chýba

Model používa 256 learned AB farebných centier. Ak codebook existuje, generovanie sa preskočí.


In [ ]:
import subprocess
import sys

if CODEBOOK.exists() and not FORCE_REBUILD_CODEBOOK:
    print("Codebook already exists. Skipping codebook generation.")
else:
    print("Generating AB codebook...")
    CODEBOOK.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "scripts/generate_ab_codebook.py",
        "--data", str(FILTERED_DATASET),
        "--out", str(CODEBOOK),
        "--image-size", "128",
        "--clusters", "256",
        "--pixels-per-image", "512",
    ]

    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

print("Codebook exists:", CODEBOOK.exists())


In [ ]:
import numpy as np

if CODEBOOK.exists():
    z = np.load(CODEBOOK)
    print("Codebook keys:", z.files)
    print("centers_ab:", z["centers_ab"].shape)
    print("weights:", z["weights"].shape)
    assert z["centers_ab"].shape == (256, 2), "Bad codebook shape"
    print("CODEBOOK OK")
else:
    raise RuntimeError("Codebook is missing.")


## 7. Tréning finálneho modelu, iba ak checkpoint chýba

Toto je hlavná ochrana proti zbytočnému trénovaniu.

Ak existuje:

```text
models/codebook_colorizer_filtered_v3/best.pt
```

tak sa tréning **preskočí**.

Ak neexistuje a `TRAIN_IF_MISSING=True`, notebook spustí rovnaký tréningový postup.


In [ ]:
import subprocess
import sys

if BEST_MODEL.exists() and not FORCE_RETRAIN:
    print("Final best model already exists. Skipping training.")
    print("Model:", BEST_MODEL)
else:
    if not TRAIN_IF_MISSING:
        raise RuntimeError("Final model is missing, but TRAIN_IF_MISSING=False.")

    print("Final model missing. Starting v3 training...")

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "scripts/train_codebook_colorizer.py",
        "--data", str(FILTERED_DATASET),
        "--codebook", str(CODEBOOK),
        "--epochs", "20",
        "--batch-size", "4",
        "--image-size", "256",
        "--num-workers", "2",
        "--amp",
        "--temperature", "0.28",
        "--lambda-ce", "2.0",
        "--lambda-ab", "6.0",
        "--chroma-weight", "8.0",
        "--model-dir", str(MODEL_DIR),
        "--sample-dir", str(SAMPLE_DIR),
    ]

    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

print("Best model exists:", BEST_MODEL.exists())


## 8. Kontrola finálneho checkpointu

Používame `best.pt`, nie `last.pt`, pretože posledná epocha nemusí byť najlepšia.


In [ ]:
import torch

if BEST_MODEL.exists():
    ckpt = torch.load(BEST_MODEL, map_location="cpu")
    print("Checkpoint:", BEST_MODEL)
    print("Epoch:", ckpt.get("epoch"))
    print("Validation loss:", ckpt.get("val_loss"))
    print("Keys:", list(ckpt.keys()))
else:
    print("Missing:", BEST_MODEL)


## 9. Final random demo

Táto časť už netrénuje. Iba načíta finálny model a vygeneruje random porovnanie:

```text
grayscale input | v3 best prediction | original target
```


In [ ]:
import subprocess
import sys
from pathlib import Path

# Create the final demo script if it is missing.
if not Path("scripts/final_v3_random_demo.py").exists():
    raise RuntimeError(
        "scripts/final_v3_random_demo.py is missing. "
        "Create it first or copy it from the final workflow."
    )

if FINAL_DEMO_GRID.exists() and not FORCE_REGENERATE_DEMO:
    print("Final demo already exists. Skipping demo generation.")
else:
    print("Generating final random demo...")
    FINAL_DEMO_DIR.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "scripts/final_v3_random_demo.py",
        "--count", "12",
        "--temperature", "0.30",
    ]

    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

print("Final demo exists:", FINAL_DEMO_GRID.exists())
print("Final demo path:", FINAL_DEMO_GRID)


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

if FINAL_DEMO_GRID.exists():
    img = Image.open(FINAL_DEMO_GRID)
    plt.figure(figsize=(12, 18))
    plt.imshow(img)
    plt.axis("off")
else:
    print("Final demo not found:", FINAL_DEMO_GRID)


## 10. Zhrnutie technického postupu

1. Pôvodný RGB baseline bol nahradený LAB workflowom.
2. Vstupom modelu je iba jasový `L` kanál.
3. Farba sa rieši cez `a,b` kanály v LAB priestore.
4. Priama regresia farieb priemerovala výsledky.
5. Preto bol použitý learned AB codebook s 256 farebnými triedami.
6. Finálny model kombinuje:
   - CNN encoder-decoder,
   - classification over AB codebook,
   - residual AB refinement.
7. Finálny checkpoint je `models/codebook_colorizer_filtered_v3/best.pt`.
8. Notebook je nastavený tak, aby preskočil tréning, ak checkpoint už existuje.
